In [20]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta, date
import time 
from sqlalchemy import text
import oracledb
import sys

In [21]:
#CONEXIONES DESTINO
DB_USER=config("USER2_BDI_DW")
DB_PASSWORD=config("PASS2_BDI_DW")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_DW")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

In [22]:
# MAESTROS GRANDES
paciente = pd.read_sql_query(f"SELECT id_paciente,per_sec FROM dim_paciente", con=connection2)
#personal = pd.read_sql_query(f"SELECT id_personal,num_doc FROM dim_personal", con=connection2)
paciente['per_sec'] = pd.to_numeric(paciente['per_sec'], errors='coerce').astype('Int64')

In [23]:
fecha_ini = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_proceso=5", con=connection2)
fecha_ini= fecha_ini.iloc[0, 0]

fecha_fin = pd.read_sql_query("SELECT fec_ter FROM etl_act where id_proceso=5", con=connection2)
fecha_fin= fecha_fin.iloc[0, 0]

In [24]:
dias_por_intervalo = 20

# Inicializa la fecha actual
fecha_actual = fecha_ini

while fecha_actual <= fecha_fin:
	inicioTiempo = time.time()
	now_inicio = datetime.now()

	fecha_ini_intervalo = fecha_actual
	fecha_fin_intervalo = fecha_actual + timedelta(days=dias_por_intervalo - 1)

	if fecha_fin_intervalo > fecha_fin:
		fecha_fin_intervalo = fecha_fin

	fecha_ini_str = fecha_ini_intervalo.strftime('%Y-%m-%d')
	fecha_fin_str = (fecha_fin_intervalo + timedelta(days=1)).strftime('%Y-%m-%d')
	fecha_fin_display_str = fecha_fin_intervalo.strftime('%Y-%m-%d')

	print(f"Procesando de {fecha_ini_str} al {fecha_fin_display_str}")

	now1 = datetime.now()
	now2 = datetime.now()

	# Restar 20 días
	now2 = now2 - timedelta(days=20)

	# Convertir a formato 'YYYY-MM-DD'
	now2 = now2.strftime('%Y-%m-%d')
	query=f"UPDATE etl_act SET fec_act ='{now1}' WHERE id_proceso=5"

	c1= text(query)
	connection2.execute(c1)

	tabla='ctaam10'
	col_tabla = "atenambatenfec"
	dat= "cext02_essi"
	col_dat='fec_ate'


	oracledb.init_oracle_client()
	oracledb.version = "8.3.0"
	sys.modules["cx_Oracle"] = oracledb
	un = config("USER4_BDI_POSTGRES")
	pw = config("PASS4_BDI_POSTGRES")
	hostname=config("HOST4_BDI_POSTGRES")
	service_name="WNET"
	port = 1527

	engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
			"host": hostname,
			"port": port,
			"service_name": service_name
		}
	)

	connection0 = engine0.connect()


	query1 = f"""
	SELECT
		c.ATENAMBCENASICOD,
		c.ATENAMBNUM,
		c.ATENAMBATENFEC,
		c.ATENAMBANAM,
		c.ATENAMBEXAMCLIN,
		c.ATENAMBTIPCONCOD,
		c.ATENAMBCSECOD,
		c.CPSCOD,
		c.ATENAMBNUMATECOD,
		c.TIPCONTLEYCOD,
		c.RESATENAMBUCOD,
		c.ATENAMBESTREGCOD,
		c.ATENAMBCREFEC,
		c.ATENAMBMODFEC,
		c.ATENAMBULTREGFEC,
		c.ATENAMBPACSECNUM,
		d.DIAGCOD,
		d.CONDDIAGCOD,
		d.ATENAMBDIAGORD,
		d.ATENAMBTIPODIAGCOD,
		d.ATENAMBCASODIAGCOD,
		d.DIAGATENAMBALTAFLAG
	FROM {tabla} c
	LEFT OUTER JOIN CTDAA10 d		 
		ON c.ATENAMBORICENASICOD = d.ATENAMBORICENASICOD
		AND c.ATENAMBCENASICOD = d.ATENAMBCENASICOD 
		AND c.ATENAMBNUM = d.ATENAMBNUM
	WHERE c.{col_tabla} >= TO_DATE('{fecha_ini_str}', 'YYYY-MM-DD') AND c.{col_tabla} < TO_DATE('{fecha_fin_str}', 'YYYY-MM-DD')
	"""

	base1 = pd.read_sql_query(query1, con=connection0)
	connection0.close()
	engine0.dispose()

	base1 = base1.rename(columns={
		'atenambcenasicod': 'cod_cas',
		'atenambnum': 'ate_num',
		'atenambatenfec': 'fec_ate',
		'atenambanam': 'ate_anam',
		'atenambexamclin': 'ate_examclin',
		'atenambtipconcod': 'cod_tipcon',
		'atenambcsecod': 'cod_cse',
		'cpscod': 'cod_cps',
		'atenambnumatecod': 'ate_cod',
		'tipcontleycod': 'cod_tipleycont',
		'resatenambucod': 'cod_resaten',
		'atenambestregcod': 'cod_est',
		'atenambcrefec': 'fec_cre',
		'atenambmodfec': 'fec_mod',
		'atenambultregfec': 'fec_ult_reg',
		'atenambpacsecnum': 'per_sec',
		'diagcod': 'cod_diag',
		'conddiagcod': 'cod_conddiag',
		'atenambdiagord': 'diag_ord',
		'atenambtipodiagcod': 'cod_tipdiag',
		'atenambcasodiagcod': 'cod_casodiag',
		'diagatenambaltaflag': 'alta_flg'
	})

	print(base1.shape)

	base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 1", con=connection2)
	
	base1['per_sec'] = pd.to_numeric(base1['per_sec'], errors='coerce').astype('Int64')

	base1=pd.merge(base1,paciente,how='left',on='per_sec')	

	base1=base1.drop('per_sec',axis=1)
	print(base1.shape)
	cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where cod_red is not null", con=connection2)
	cas = cas.dropna()
	cas['cod_cas'] = cas['cod_cas'].str.strip()
	base1['cod_cas'] = base1['cod_cas'].str.strip()
	base1=pd.merge(base1,cas,how='left',on='cod_cas')

	base1=base1.drop('cod_cas',axis=1)
	
	tipcon = pd.read_sql_query(f"SELECT id_tipcon,cod_tipcon FROM dim_tipcon", con=connection2)
	tipcon['cod_tipcon'] = tipcon['cod_tipcon'].str.strip()
	base1['cod_tipcon'] = base1['cod_tipcon'].str.strip()
	base1=pd.merge(base1,tipcon,how='left',on='cod_tipcon')

	base1=base1.drop('cod_tipcon',axis=1)

	csep = pd.read_sql_query(f"SELECT id_csep,cod_cse FROM dim_csep", con=connection2)
	csep['cod_cse'] = csep['cod_cse'].str.strip()
	base1['cod_cse'] = base1['cod_cse'].str.strip()
	base1=pd.merge(base1,csep,how='left',on='cod_cse')

	base1=base1.drop('cod_cse',axis=1)

	cps = pd.read_sql_query(f"SELECT id_cps,cod_cps FROM dim_cps", con=connection2)
	cps['cod_cps'] = cps['cod_cps'].str.strip()
	base1['cod_cps'] = base1['cod_cps'].str.strip()
	base1=pd.merge(base1,cps,how='left',on='cod_cps')

	base1=base1.drop('cod_cps',axis=1)

	ley = pd.read_sql_query(f"SELECT id_tipleycont,cod_tipleycont FROM dim_tipleycont", con=connection2)
	ley['cod_tipleycont'] = ley['cod_tipleycont'].str.strip()
	base1['cod_tipleycont'] = base1['cod_tipleycont'].str.strip()
	base1=pd.merge(base1,ley,how='left',on='cod_tipleycont')

	base1=base1.drop('cod_tipleycont',axis=1)

	resaten = pd.read_sql_query(f"SELECT id_resaten,cod_resaten FROM dim_resaten", con=connection2)
	resaten['cod_resaten'] = resaten['cod_resaten'].str.strip()
	base1['cod_resaten'] = base1['cod_resaten'].str.strip()
	base1=pd.merge(base1,resaten,how='left',on='cod_resaten')

	base1=base1.drop('cod_resaten',axis=1)

	estreg = pd.read_sql_query(f"SELECT id_estreg,cod_est FROM dim_estreg", con=connection2)
	estreg['cod_est'] = estreg['cod_est'].str.strip()
	base1['cod_est'] = base1['cod_est'].str.strip()
	base1=pd.merge(base1,estreg,how='left',on='cod_est')

	base1=base1.drop('cod_est',axis=1)

	diagcod = pd.read_sql_query(f"SELECT id_cie,cod_diag FROM dim_cie10", con=connection2)
	diagcod['cod_diag'] = diagcod['cod_diag'].str.strip()
	base1['cod_diag'] = base1['cod_diag'].str.strip()
	base1=pd.merge(base1,diagcod,how='left',on='cod_diag')

	base1=base1.drop('cod_diag',axis=1)

	conddiag = pd.read_sql_query(f"SELECT id_conddiag,cod_conddiag FROM dim_conddiag", con=connection2)
	conddiag['cod_conddiag'] = conddiag['cod_conddiag'].str.strip()
	base1['cod_conddiag'] = base1['cod_conddiag'].str.strip()
	base1=pd.merge(base1,conddiag,how='left',on='cod_conddiag')

	base1=base1.drop('cod_conddiag',axis=1)

	tipodiag = pd.read_sql_query(f"SELECT id_tipdiag,cod_tipdiag FROM dim_tipdiag", con=connection2)
	tipodiag['cod_tipdiag'] = tipodiag['cod_tipdiag'].str.strip()
	base1['cod_tipdiag'] = base1['cod_tipdiag'].str.strip()
	base1=pd.merge(base1,tipodiag,how='left',on='cod_tipdiag')

	base1=base1.drop('cod_tipdiag',axis=1)

	casodiag = pd.read_sql_query(f"SELECT id_casodiag,cod_casodiag FROM dim_casodiag", con=connection2)
	casodiag['cod_casodiag'] = casodiag['cod_casodiag'].str.strip()
	base1['cod_casodiag'] = base1['cod_casodiag'].str.strip()
	base1=pd.merge(base1,casodiag,how='left',on='cod_casodiag')

	base1=base1.drop('cod_casodiag',axis=1)

	print(base1.shape)
	
	df1_columns = set(base1.columns)
	df2_columns = set(base2.columns) 
	different_columns = df2_columns - df1_columns
	different_columns

	borrando = f"DELETE FROM {dat} WHERE {col_dat} >='{fecha_ini_str}' and {col_dat} <'{fecha_fin_str}'"
	borrado = connection2.execute(borrando)

	comunes = set(base1.columns).intersection(set(base2.columns)) 
	base1 = base1[list(comunes)]
	base1.to_sql(name=f'{dat}', con=engine2, if_exists='append', index=False,chunksize=500000)

	fecha_actual = fecha_fin_intervalo + timedelta(days=1)

	finproceso = time.time()
	tiempoproceso = finproceso - inicioTiempo
	tiempoproceso = round(tiempoproceso, 3)
	print("Proceso completado satisfactoriamente en " + str(tiempoproceso) + " segundos")

query2=f"UPDATE etl_act SET fec_ini ='{now2}' WHERE id_proceso=5"
c2= text(query2)
connection2.execute(c2)

Procesando de 2024-08-16 al 2024-09-04
(2078637, 22)
(2078641, 22)
(2081253, 22)
Proceso completado satisfactoriamente en 842.457 segundos
Procesando de 2024-09-05 al 2024-09-24
(172375, 22)
(172375, 22)
(172738, 22)
Proceso completado satisfactoriamente en 164.019 segundos
Procesando de 2024-09-25 al 2024-10-14
(0, 22)
(0, 22)
(0, 22)
Proceso completado satisfactoriamente en 0.474 segundos
Procesando de 2024-10-15 al 2024-11-03
(0, 22)
(0, 22)
(0, 22)
Proceso completado satisfactoriamente en 0.262 segundos
Procesando de 2024-11-04 al 2024-11-23
(0, 22)
(0, 22)
(0, 22)
Proceso completado satisfactoriamente en 0.26 segundos
Procesando de 2024-11-24 al 2024-12-13
(0, 22)
(0, 22)
(0, 22)
Proceso completado satisfactoriamente en 0.232 segundos
Procesando de 2024-12-14 al 2025-01-01
(0, 22)
(0, 22)
(0, 22)
Proceso completado satisfactoriamente en 0.226 segundos


In [25]:
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()